# MZ세대 공무원 이직의도: Gradient Boosting과 SHAP

- 2025 한국심리측정평가학회 춘계학술대회 구두 발표(SHAP 기반 설명 가능한 AI로 들여다본 공무원 이직의도의 심리적 메커니즘) 분석
- 자료: 2023 공직생활실태조사 중 MZ세대 공무원 4,288명, 예측변수 55개, 결과변수는 이직의도(있음/없음)
- 입력 파일: `data_cleaned.csv` (ml-practice 저장소의 `turnover-logistic-ml/01_logistic_regression.Rmd`에서 생성, 원자료는 비공개)
- 순서: PyCaret 모형 비교 → 최종 Gradient Boosting 모형 → 시험 자료 성능 → 변수 중요도 → SHAP 해석
- 시험 자료 성능: 정확도 .734, AUC .819

## 1. 준비

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from pycaret.classification import setup, compare_models, pull
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
from sklearn.model_selection import train_test_split

data = pd.read_csv("data_cleaned.csv")
print(data.shape)
print(data["Turnover_Intention"].value_counts())  # 1 = 이직의도 있음

## 2. 분류 모형 비교 (PyCaret)

- 기본 설정의 분류 모형 전체를 10겹 층화 교차검증으로 비교 (AUC 기준 정렬)
- 비교에서 가장 성능이 높은 Gradient Boosting을 최종 모형으로 선택

In [ ]:
s = setup(data, target="Turnover_Intention", session_id=1, fold=10,
          fold_strategy="stratifiedkfold", n_jobs=-1)
best_model = compare_models(sort="AUC")
print(pull())

## 3. 최종 모형: Gradient Boosting

- 학습 70%, 시험 30% 층화 분할
- 하이퍼파라미터: 튜닝으로 정한 값 (트리 260개, 학습률 0.05, 최대 깊이 4 등)

In [ ]:
X = data.drop("Turnover_Intention", axis=1)
y = data["Turnover_Intention"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, train_size=0.7, stratify=y, random_state=1)

gbc = GradientBoostingClassifier(
    n_estimators=260,
    learning_rate=0.05,
    max_depth=4,
    min_samples_split=4,
    min_samples_leaf=2,
    min_impurity_decrease=0.05,
    subsample=0.8,
    max_features="sqrt",
    random_state=1,
)
gbc.fit(X_train, y_train)

# 시험 자료 성능
y_prob = gbc.predict_proba(X_test)[:, 1]
y_pred = gbc.predict(X_test)
print(f"정확도: {accuracy_score(y_test, y_pred):.4f}")
print(f"AUC: {roc_auc_score(y_test, y_prob):.4f}")
print(classification_report(y_test, y_pred))

## 4. 변수 중요도

- 불순도 감소 기준 상위 10개 변수 (전체 중요도의 약 52%)

In [ ]:
importance = pd.Series(gbc.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importance.head(10).round(3))
print(f"상위 10개 비중: {importance.head(10).sum():.2%}")

importance.head(10)[::-1].plot.barh(figsize=(7, 5), color="steelblue")
plt.xlabel("Importance")
plt.tight_layout()
plt.show()

## 5. SHAP 해석

- 시험 자료 1,287명의 SHAP 값 계산 (값의 단위는 이직의도 로그오즈 기여도)
- 평균 |SHAP| 상위 10개 변수의 중요도, 방향, 의존성 그림 확인

In [ ]:
explainer = shap.TreeExplainer(gbc)
shap_values = explainer.shap_values(X_test)

mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=X.columns).sort_values(ascending=False)
print(mean_abs_shap.head(10).round(4))
print(f"상위 10개 비중: {mean_abs_shap.head(10).sum() / mean_abs_shap.sum():.2%}")

shap.summary_plot(shap_values, X_test, plot_type="bar", max_display=10)  # 평균 |SHAP| 중요도
shap.summary_plot(shap_values, X_test, max_display=10)                   # 사례별 분포와 방향

In [ ]:
# 상위 10개 변수의 의존성 그림 (색: 상호작용이 가장 큰 변수)
for feature in mean_abs_shap.index[:10]:
    shap.dependence_plot(feature, shap_values, X_test)